In [1]:
%pip install nflreadpy


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install pyarrow


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import nflreadpy as nfl

# Load all available NFL team statistics
team_stats = nfl.load_team_stats(seasons=True)

# Inspect the data
print(team_stats.shape)
print(team_stats.head())
print(team_stats.columns)

(14833, 138)
shape: (5, 138)
┌────────┬──────┬──────┬─────────────┬───┬─────────────┬──────────────┬──────────────┬─────────────┐
│ season ┆ week ┆ team ┆ season_type ┆ … ┆ pt_returned ┆ pt_return_ya ┆ pt_return_td ┆ pt_net_yard │
│ ---    ┆ ---  ┆ ---  ┆ ---         ┆   ┆ ---         ┆ rds          ┆ s            ┆ s           │
│ i32    ┆ i32  ┆ str  ┆ str         ┆   ┆ i32         ┆ ---          ┆ ---          ┆ ---         │
│        ┆      ┆      ┆             ┆   ┆             ┆ i32          ┆ i32          ┆ i32         │
╞════════╪══════╪══════╪═════════════╪═══╪═════════════╪══════════════╪══════════════╪═════════════╡
│ 1999   ┆ 1    ┆ ARI  ┆ REG         ┆ … ┆ 1           ┆ 2            ┆ 0            ┆ 170         │
│ 1999   ┆ 1    ┆ ATL  ┆ REG         ┆ … ┆ 1           ┆ -1           ┆ 0            ┆ 89          │
│ 1999   ┆ 1    ┆ BUF  ┆ REG         ┆ … ┆ 0           ┆ 0            ┆ 0            ┆ 166         │
│ 1999   ┆ 1    ┆ CAR  ┆ REG         ┆ … ┆ 5           ┆ 31   

In [ ]:
# Load NFL schedules
schedules = nfl.load_schedules(seasons=True)

# Convert both datasets from to pandas
team_stats = team_stats.to_pandas()
schedules = schedules.to_pandas()

print("Team stats type:", type(team_stats))
print("Schedules type:", type(schedules))

Team stats type: <class 'pandas.core.frame.DataFrame'>
Schedules type: <class 'pandas.core.frame.DataFrame'>


In [ ]:
# Split the data into training, validation, and test sets based on seasons
seasons = sorted(team_stats["season"].unique())

n = len(seasons)

train_seasons = seasons[:int(n * 0.70)]
validation_seasons = seasons[int(n * 0.70):int(n * 0.85)]
test_seasons = seasons[int(n * 0.85):]

print("Training:", train_seasons)
print("Validation:", validation_seasons)
print("Testing:", test_seasons)

Training: [np.int32(1999), np.int32(2000), np.int32(2001), np.int32(2002), np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017)]
Validation: [np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021)]
Testing: [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


In [19]:
# Check that team_stats is a DataFrame
print("Team stats type:", type(team_stats))

# Check the columns available in each dataset
print("Team stats columns needed:")
print([c for c in ["season", "week", "team", "game_id", "opponent_team"]
       if c in team_stats.columns])

print("\nSchedule columns needed:")
print([c for c in ["game_id", "season", "week", "home_team", "away_team",
                  "home_score", "away_score"]
       if c in schedules.columns])

# Required columns
required_team_cols = {"season", "week", "team", "game_id"}
required_schedule_cols = {
    "game_id", "home_team", "away_team", "home_score", "away_score"
}

# Identify missing columns
missing_team = required_team_cols - set(team_stats.columns)
missing_schedule = required_schedule_cols - set(schedules.columns)

if missing_team:
    raise ValueError(f"team_stats is missing required columns: {missing_team}")

if missing_schedule:
    raise ValueError(f"schedules is missing required columns: {missing_schedule}")

# Make sure join keys have consistent types
team_stats["game_id"] = team_stats["game_id"].astype(str)
schedules["game_id"] = schedules["game_id"].astype(str)

# Keep one schedule row per game
schedules = schedules.drop_duplicates(subset="game_id").copy()

# Convert scores to numeric
for col in ["home_score", "away_score"]:
    schedules[col] = pd.to_numeric(schedules[col], errors="coerce")

# Check for duplicate games
print("Duplicate game_id rows in schedules:",
      schedules["game_id"].duplicated().sum())

Team stats type: <class 'pandas.core.frame.DataFrame'>
Team stats columns needed:
['season', 'week', 'team', 'game_id', 'opponent_team']

Schedule columns needed:
['game_id', 'season', 'week', 'home_team', 'away_team', 'home_score', 'away_score']
Duplicate game_id rows in schedules: 0


In [8]:
# Select the schedule columns needed to identify game results
schedule_cols = [
    "game_id", "home_team", "away_team", "home_score", "away_score"
]

# Include optional schedule columns only if they exist
# Avoid adding season/week if they already exist in team_stats
for optional_col in ["game_type", "season_type", "season", "week", "gameday"]:
    if (optional_col in schedules.columns
            and optional_col not in team_stats.columns
            and optional_col not in schedule_cols):
        schedule_cols.append(optional_col)

# Merge team statistics with game schedules using game_id
games = team_stats.merge(
    schedules[schedule_cols],
    on="game_id",
    how="left",
    validate="many_to_one"
)

# Identify whether each team played at home or away
is_home = games["team"].eq(games["home_team"])
is_away = games["team"].eq(games["away_team"])

games["is_home"] = is_home.astype(int)

# Calculate points scored and points allowed for each team
games["points_for"] = np.where(
    is_home, games["home_score"], games["away_score"]
)

games["points_against"] = np.where(
    is_home, games["away_score"], games["home_score"]
)

# Keep only rows that match a team in the schedule
games = games[is_home | is_away].copy()

# Remove games that have not been completed
games = games.dropna(
    subset=["points_for", "points_against"]
).copy()

# Calculate point differential
games["point_differential"] = (
    games["points_for"] - games["points_against"]
)

# Exclude ties because our target is binary win/loss
games = games[games["point_differential"] != 0].copy()

# Create the target variable:
# 1 = Win, 0 = Loss
games["win"] = (
    games["point_differential"] > 0
).astype(int)

# Display results
print("Merged rows:", len(games))
print("Unique games:", games["game_id"].nunique())

print("\nRows per game (should generally be 2):")
display(
    games.groupby("game_id")
    .size()
    .value_counts()
    .sort_index()
)

print("\nSample of merged data:")
display(
    games[[
        "season", "week", "team", "opponent_team",
        "is_home", "points_for", "points_against",
        "point_differential", "win"
    ]].head(10)
)

Merged rows: 13879
Unique games: 7363

Rows per game (should generally be 2):


1     847
2    6516
Name: count, dtype: int64


Sample of merged data:


,season,week,team,opponent_team,is_home,points_for,points_against,point_differential,win
0,1999,1,ARI,PHI,0,25.0,24.0,1.0,1
1,1999,1,ATL,MIN,1,14.0,17.0,-3.0,0
2,1999,1,BUF,IND,0,14.0,31.0,-17.0,0
3,1999,1,CAR,NO,0,10.0,19.0,-9.0,0
4,1999,1,CHI,KC,1,20.0,17.0,3.0,1
5,1999,1,CIN,TEN,0,35.0,36.0,-1.0,0
6,1999,1,CLE,PIT,1,0.0,43.0,-43.0,0
7,1999,1,DAL,WAS,0,41.0,35.0,6.0,1
8,1999,1,DEN,MIA,1,21.0,38.0,-17.0,0
9,1999,1,DET,SEA,0,28.0,20.0,8.0,1


In [9]:
# Keep only regular-season games
if "game_type" in games.columns:
    games = games[
        games["game_type"].astype(str).str.upper() == "REG"
    ].copy()

elif "season_type" in games.columns:
    games = games[
        games["season_type"].astype(str).str.upper().isin(
            ["REG", "REGULAR"]
        )
    ].copy()

else:
    print("WARNING: No game_type or season_type column found. "
          "Regular-season filtering was not applied.")

# Display results
print("Rows after regular-season filter:", len(games))
print("Seasons available:", sorted(games["season"].dropna().unique()))

Rows after regular-season filter: 13289
Seasons available: [np.int32(1999), np.int32(2000), np.int32(2001), np.int32(2002), np.int32(2003), np.int32(2004), np.int32(2005), np.int32(2006), np.int32(2007), np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]


In [10]:
# Base features selected from the team_stats columns you provided.
base_feature_candidates = [
    "passing_yards",
    "passing_tds",
    "passing_interceptions",
    "passing_epa",
    "rushing_yards",
    "rushing_tds",
    "rushing_epa",
    "receiving_yards",
    "def_sacks",
    "def_interceptions",
    "def_tds",
    "fg_pct",
    "pt_net_yards",
    "fumbles_lost_total",
]

# Keep only features that exist in the downloaded file.
base_features = [c for c in base_feature_candidates if c in games.columns]
missing_features = [c for c in base_feature_candidates if c not in games.columns]

print("Features included:", base_features)
print("Features not found (skipped):", missing_features)

# Convert selected columns to numeric.
for col in base_features:
    games[col] = pd.to_numeric(games[col], errors="coerce")

# Additional features using columns that exist.
if {"passing_yards", "rushing_yards"}.issubset(games.columns):
    games["total_offensive_yards"] = games["passing_yards"] + games["rushing_yards"]
    base_features.append("total_offensive_yards")

if {"passing_tds", "rushing_tds", "special_teams_tds", "def_tds"}.issubset(games.columns):
    games["total_team_tds"] = (
        games["passing_tds"] + games["rushing_tds"]
        + games["special_teams_tds"] + games["def_tds"]
    )
    base_features.append("total_team_tds")

# Approximate turnover margin: defensive takeaways minus offensive turnovers.
turnover_components = {
    "def_interceptions", "fumble_recovery_opp",
    "passing_interceptions", "fumbles_lost_total"
}
if turnover_components.issubset(games.columns):
    games["turnover_margin"] = (
        games["def_interceptions"].fillna(0)
        + games["fumble_recovery_opp"].fillna(0)
        - games["passing_interceptions"].fillna(0)
        - games["fumbles_lost_total"].fillna(0)
    )
    base_features.append("turnover_margin")

# Remove any accidental duplicate feature names while preserving order.
base_features = list(dict.fromkeys(base_features))

print("Final number of per-game input statistics:", len(base_features))

Features included: ['passing_yards', 'passing_tds', 'passing_interceptions', 'passing_epa', 'rushing_yards', 'rushing_tds', 'rushing_epa', 'receiving_yards', 'def_sacks', 'def_interceptions', 'def_tds', 'fg_pct', 'pt_net_yards', 'fumbles_lost_total']
Features not found (skipped): []
Final number of per-game input statistics: 17


In [ ]:
# Settings for the remaining steps
from pathlib import Path

# Number of previous games used to calculate rolling averages
ROLLING_WINDOW = 3

# Minimum previous games required before a row can be used
MIN_PRIOR_GAMES = 3

# Folder for the prepared files
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Create lagged rolling averages

games = games.sort_values(["team", "season", "week", "game_id"]).copy()

rolling_feature_names = []

for feature in base_features:
    new_col = f"{feature}_avg_last_{ROLLING_WINDOW}"
    games[new_col] = (
        games.groupby("team")[feature]
        .transform(
            lambda s: s.shift(1).rolling(
                window=ROLLING_WINDOW,
                min_periods=ROLLING_WINDOW
            ).mean()
        )
    )
    rolling_feature_names.append(new_col)

# Number of completed games the team has played before this one
games["prior_games_played"] = games.groupby("team").cumcount()

# Team's win rate before the current game (shifted so the current result is excluded)
games["prior_wins"] = (
    games.groupby("team")["win"].transform(lambda s: s.shift(1).expanding().sum())
)
games["prior_win_rate"] = (
    games["prior_wins"] / games["prior_games_played"].replace(0, np.nan)
)

# Keep rows with enough history, then drop rows missing any rolling feature
model_data = games[games["prior_games_played"] >= MIN_PRIOR_GAMES].copy()

rows_before = len(model_data)
model_data = model_data.dropna(subset=rolling_feature_names).copy()
print("Rows dropped for missing rolling features:", rows_before - len(model_data))

print("Prepared rows:", len(model_data))
print("Number of unique games:", model_data["game_id"].nunique())
display(model_data[[
    "season", "week", "team", "opponent_team", "is_home",
    "prior_games_played", "prior_win_rate", "win"
] + rolling_feature_names[:5]].head(10))

Rows dropped for missing rolling features: 4483
Prepared rows: 8710
Number of unique games: 6026


,season,week,team,opponent_team,is_home,prior_games_played,prior_win_rate,win,passing_yards_avg_last_3,passing_tds_avg_last_3,passing_interceptions_avg_last_3,passing_epa_avg_last_3,rushing_yards_avg_last_3
86,1999,4,ARI,DAL,0,3,0.333333,0,196.000000,0.333333,3.000000,-9.652737,81.000000
226,1999,9,ARI,NYJ,0,7,0.285714,0,174.666667,0.666667,1.666667,-10.720215,58.000000
345,1999,13,ARI,PHI,1,11,0.454545,1,191.666667,1.000000,0.666667,-4.047452,110.000000
603,2000,4,ARI,GB,1,18,0.388889,0,319.000000,2.000000,2.000000,6.306703,73.666667
631,2000,5,ARI,SF,0,19,0.368421,0,250.000000,1.333333,2.333333,1.308791,56.333333
659,2000,6,ARI,CLE,1,20,0.350000,1,223.666667,1.000000,1.333333,1.643274,84.000000
685,2000,7,ARI,PHI,1,21,0.380952,0,199.666667,1.000000,1.333333,-2.148952,100.000000
979,2000,17,ARI,WAS,0,31,0.290323,0,241.666667,1.333333,1.333333,-1.523984,54.333333
1060,2001,2,ARI,DEN,1,32,0.281250,0,197.000000,0.666667,2.000000,-9.558393,65.000000
1087,2001,3,ARI,ATL,1,33,0.272727,0,217.333333,0.666667,2.000000,-8.526896,79.666667


In [13]:
# Check the prepared data

print("Target counts:")
display(model_data["win"].value_counts().rename(index={0: "Loss", 1: "Win"}))

print("\nTarget proportions:")
display(model_data["win"].value_counts(normalize=True).rename(index={0: "Loss", 1: "Win"}))

print("\nMissing values in model features:")
display(model_data[rolling_feature_names + ["is_home", "prior_win_rate"]].isna().sum())

print("\nDuplicate (game_id, team) rows:",
      model_data.duplicated(subset=["game_id", "team"]).sum())

print("\nTarget values:", sorted(model_data["win"].unique().tolist()))

print("\nRows per season:")
display(model_data.groupby("season").size())

Target counts:


win
Win     4462
Loss    4248
Name: count, dtype: int64


Target proportions:


win
Win     0.512285
Loss    0.487715
Name: proportion, dtype: float64


Missing values in model features:


passing_yards_avg_last_3            0
passing_tds_avg_last_3              0
passing_interceptions_avg_last_3    0
passing_epa_avg_last_3              0
rushing_yards_avg_last_3            0
rushing_tds_avg_last_3              0
rushing_epa_avg_last_3              0
receiving_yards_avg_last_3          0
def_sacks_avg_last_3                0
def_interceptions_avg_last_3        0
def_tds_avg_last_3                  0
fg_pct_avg_last_3                   0
pt_net_yards_avg_last_3             0
fumbles_lost_total_avg_last_3       0
total_offensive_yards_avg_last_3    0
total_team_tds_avg_last_3           0
turnover_margin_avg_last_3          0
is_home                             0
prior_win_rate                      0
dtype: int64


Duplicate (game_id, team) rows: 0

Target values: [0, 1]

Rows per season:


season
1999    247
2000    263
2001    303
2002    291
2003    285
2004    292
2005    295
2006    287
2007    282
2008    302
2009    296
2010    282
2011    335
2012    317
2013    322
2014    295
2015    277
2016    310
2017    339
2018    343
2019    337
2020    341
2021    335
2022    386
2023    374
2024    377
2025    381
2026    216
dtype: int64

In [ ]:
# Coverage check

games_per_team_season = games.groupby(["season", "team"]).size()

print("Teams per season (should be 28-32):")
display(games.groupby("season")["team"].nunique().describe())

low_coverage = games_per_team_season[games_per_team_season < 14]
print("\nTeam-seasons with fewer than 14 games:", len(low_coverage))
if len(low_coverage) > 0:
    display(low_coverage.head(20))

Teams per season (should be 28-32):


count    28.000000
mean     29.892857
std       1.448864
min      28.000000
25%      29.000000
50%      29.000000
75%      31.250000
max      32.000000
Name: team, dtype: float64


Team-seasons with fewer than 14 games: 34


season  team
2001    JAX      8
2002    JAX      8
2026    ARI     10
        ATL     10
        BAL     10
        BUF      9
        CAR      9
        CHI      9
        CIN      9
        CLE     10
        DAL     10
        DEN      9
        DET      9
        GB      10
        HOU     10
        IND     11
        JAX      9
        KC       9
        LA      10
        LAC      9
dtype: int64

In [ ]:
# Create chronological train, validation, and test sets

train_data = model_data[model_data["season"].isin(train_seasons)].copy()
validation_data = model_data[model_data["season"].isin(validation_seasons)].copy()
test_data = model_data[model_data["season"].isin(test_seasons)].copy()

print("Train seasons:", sorted(train_data["season"].unique().tolist()))
print("Validation seasons:", sorted(validation_data["season"].unique().tolist()))
print("Test seasons:", sorted(test_data["season"].unique().tolist()))

print("\nSplit sizes:")
print("Train:", train_data.shape)
print("Validation:", validation_data.shape)
print("Test:", test_data.shape)

# Splits must be non-empty, non-overlapping, and strictly ordered in time
if train_data.empty or validation_data.empty or test_data.empty:
    print("\nWARNING: At least one split is empty. Check train_seasons, "
          "validation_seasons, and test_seasons.")
else:
    assert train_data["season"].max() < validation_data["season"].min()
    assert validation_data["season"].max() < test_data["season"].min()
    assert not (set(train_data["game_id"]) & set(validation_data["game_id"]))
    assert not (set(validation_data["game_id"]) & set(test_data["game_id"]))
    print("\nChronological split checks passed.")

print("\nWin rate by split:")
display(pd.DataFrame({
    "train": train_data["win"].mean(),
    "validation": validation_data["win"].mean(),
    "test": test_data["win"].mean(),
}, index=["win_rate"]))

Train seasons: [1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017]
Validation seasons: [2018, 2019, 2020, 2021]
Test seasons: [2022, 2023, 2024, 2025, 2026]

Split sizes:
Train: (5620, 172)
Validation: (1356, 172)
Test: (1734, 172)

Chronological split checks passed.

Win rate by split:


,train,validation,test
win_rate,0.51548,0.522124,0.494233


In [ ]:
# Define modeling columns and save outputs `win` is the target. 
# Predictors are the rolling averages, home/away status, and prior win rate.

feature_columns = rolling_feature_names + ["is_home", "prior_win_rate"]
target_column = "win"

# Identifiers and same-game outcomes kept for interpretation only (NOT predictors)
identifier_columns = [
    "season", "week", "game_id", "team", "opponent_team",
    "is_home", "points_for", "points_against", "point_differential",
    "prior_games_played", "prior_wins", "prior_win_rate", "win"
]

export_columns = list(dict.fromkeys(
    [c for c in identifier_columns if c in model_data.columns] + feature_columns
))

model_data[export_columns].to_csv(OUTPUT_DIR / "nfl_team_game_model_data.csv", index=False)
train_data[export_columns].to_csv(OUTPUT_DIR / "nfl_train.csv", index=False)
validation_data[export_columns].to_csv(OUTPUT_DIR / "nfl_validation.csv", index=False)
test_data[export_columns].to_csv(OUTPUT_DIR / "nfl_test.csv", index=False)

# Document the predictors
metadata = pd.DataFrame({
    "feature": feature_columns,
    "description": (
        [f"Mean of previous {ROLLING_WINDOW} games for {f}" for f in base_features]
        + ["1 if team is home, otherwise 0", "Team's win rate before this game"]
    )
})
metadata.to_csv(OUTPUT_DIR / "feature_descriptions.csv", index=False)

print("Saved files to:", OUTPUT_DIR.resolve())
print("Number of predictors:", len(feature_columns))
print(feature_columns)

Saved files to: /Users/bradydaylor/Desktop/2026-2027/Fall 2026/DS4420/ds4420project/Data/data/processed
Number of predictors: 19
['passing_yards_avg_last_3', 'passing_tds_avg_last_3', 'passing_interceptions_avg_last_3', 'passing_epa_avg_last_3', 'rushing_yards_avg_last_3', 'rushing_tds_avg_last_3', 'rushing_epa_avg_last_3', 'receiving_yards_avg_last_3', 'def_sacks_avg_last_3', 'def_interceptions_avg_last_3', 'def_tds_avg_last_3', 'fg_pct_avg_last_3', 'pt_net_yards_avg_last_3', 'fumbles_lost_total_avg_last_3', 'total_offensive_yards_avg_last_3', 'total_team_tds_avg_last_3', 'turnover_margin_avg_last_3', 'is_home', 'prior_win_rate']


In [17]:
# Final leakage check
# 1) Rebuild one team's rolling averages by hand and compare to the notebook values.
# 2) Show one team's games in order so the shift is easy to see.

example_team = model_data["team"].dropna().iloc[0]

team_games = (
    games[games["team"] == example_team]
    .sort_values(["season", "week", "game_id"])
    .reset_index(drop=True)
)

checked, mismatches = 0, 0
for feature, roll_col in zip(base_features[:4], rolling_feature_names[:4]):
    for i in range(ROLLING_WINDOW, len(team_games)):
        manual = team_games[feature].iloc[i - ROLLING_WINDOW:i].mean(skipna=False)  # games BEFORE row i
        actual = team_games[roll_col].iloc[i]
        if not (np.isnan(manual) and np.isnan(actual)) and not np.isclose(manual, actual):
            mismatches += 1
        checked += 1

print(f"Manual rolling-average check for {example_team}: "
      f"{checked} values checked, {mismatches} mismatches")
assert mismatches == 0, "Rolling features do not match the previous games only."

# No same-game outcome columns should be in the predictor list
forbidden = {"points_for", "points_against", "point_differential", "win"}
assert forbidden.isdisjoint(feature_columns), "Same-game outcome found in features."
print("No same-game outcome columns in feature_columns.")

display(
    model_data[model_data["team"] == example_team]
    .sort_values(["season", "week", "game_id"])[
        ["season", "week", "team", "points_for", "points_against",
         "point_differential", "win", "prior_win_rate"] + rolling_feature_names[:4]
    ].head(12)
)

print("Do not put these columns in X:", sorted(forbidden))

Manual rolling-average check for ARI: 1768 values checked, 0 mismatches
No same-game outcome columns in feature_columns.


,season,week,team,points_for,points_against,point_differential,win,prior_win_rate,passing_yards_avg_last_3,passing_tds_avg_last_3,passing_interceptions_avg_last_3,passing_epa_avg_last_3
86,1999,4,ARI,7.0,35.0,-28.0,0,0.333333,196.000000,0.333333,3.000000,-9.652737
226,1999,9,ARI,7.0,12.0,-5.0,0,0.285714,174.666667,0.666667,1.666667,-10.720215
345,1999,13,ARI,21.0,17.0,4.0,1,0.454545,191.666667,1.000000,0.666667,-4.047452
603,2000,4,ARI,3.0,29.0,-26.0,0,0.388889,319.000000,2.000000,2.000000,6.306703
631,2000,5,ARI,20.0,27.0,-7.0,0,0.368421,250.000000,1.333333,2.333333,1.308791
659,2000,6,ARI,29.0,21.0,8.0,1,0.350000,223.666667,1.000000,1.333333,1.643274
685,2000,7,ARI,14.0,33.0,-19.0,0,0.380952,199.666667,1.000000,1.333333,-2.148952
979,2000,17,ARI,3.0,20.0,-17.0,0,0.290323,241.666667,1.333333,1.333333,-1.523984
1060,2001,2,ARI,17.0,38.0,-21.0,0,0.281250,197.000000,0.666667,2.000000,-9.558393
1087,2001,3,ARI,14.0,34.0,-20.0,0,0.272727,217.333333,0.666667,2.000000,-8.526896


Do not put these columns in X: ['point_differential', 'points_against', 'points_for', 'win']
